# Lab 3, part B: the agent

Lab 3 (A+B) costs: $0.134

**Two halves, and the whole lab is about telling them apart.**

Part A built four tools, two gates, one normalising pass and a handoff contract, and proved
every one of them offline. This part hands them to an agent.

The gates are code, so they hold on every run and there is nothing to measure. The
decisions, resolve or escalate and ask or assume, are the model's, so they have a rate. This
part shows the first half holding, then reports the second half as the number it is.

**This part calls the API.** Every run caps its turns and its spend, and all of it runs on
the smaller model, which is the right size for tool routing over four tools.

Setup, once, in the folder above this one: copy `.env.example` to `.env` and read the notes
at the top of it. On a Claude subscription leave the credential lines blank and run `claude`
once to sign in; on API billing put a key in `ANTHROPIC_API_KEY`. The cell below prints
which of the two it found.

In [ ]:
import labkit

lab = labkit.start(model_env="LAB_MODEL_SMALL",
                   needs=["data/customers.json"],
                   needs_hint="Run part A first: it writes the data these tools read.")

import support.hooks as H
import support.tools as T

## 1. The agent, and nothing it did not get from us

**Four options carry the whole configuration.**

| Option | What it does here |
|---|---|
| `mcp_servers={"support": ...}` | the in-process server from part A, which is why the tools are named `mcp__support__*` |
| `tools=[]` | removes every built-in. No `Bash`, no `Read`: the agent has our four tools and nothing else |
| `setting_sources=[]` | loads nothing from this machine. No CLAUDE.md, no settings, no skills |
| `max_turns`, `max_budget_usd` | the safety net on a loop that decides its own length |

`converse()` runs a `ClaudeSDKClient` rather than a one-shot `query()`, because a support
conversation is not one request. The customer answers, and what the agent does next depends
on what they said. One of the six conversations below turns entirely on the second thing the
customer says.

**The system prompt carries the judgement half**: three escalation triggers stated as
criteria, and a few examples aimed at the boundary rather than at the obvious cases. Its
prefix is identical for every conversation, so the six runs read cache rather than paying
for the prompt six times.

**Note what is not in it.** There is no instruction to verify the customer first, and no
mention of the 500 pound limit. Those are the gates, and this part exists to show that
moving them out of the prompt is what makes them hold.

In [ ]:
from claude_agent_sdk import HookMatcher

SYSTEM = """
You are a customer support resolution agent. Resolve what you can, escalate what you must,
and be right about which is which.

<escalation_criteria>
Escalate immediately when: the customer asks for a human or a manager, and do not
investigate first; the request is not covered by policy, for example competitor price
matching, when policy covers our own site only; two attempts have not produced progress.
Resolve when policy covers the request and the tools can complete it, even if the customer
is frustrated. Acknowledge the frustration, then offer the resolution, and escalate only if
they ask again.
Ask for another identifier, an email address, a phone number or an order number, when
get_customer returns more than one match. Never choose between candidates yourself.
</escalation_criteria>

<examples>
Customer: "This is outrageous, I am very unhappy with the quality."
-> Acknowledge, then offer a replacement or a refund. This is not an escalation.
Customer: "No, I want to talk to someone."
-> Call escalate_to_human now.
Customer: "A competitor has this 30% cheaper, give me a discount."
-> Escalate: policy is silent on competitor price matching.
</examples>

When a tool fails, read errorCategory and isRetryable. Retry a transient failure once.
Never retry a business refusal: explain it to the customer in their own terms and offer
what policy does allow.
Keep identifiers, amounts and dates exactly as the tools reported them.
""".strip()

# Which hook runs on which tool. A matcher is a regex over the tool name, and a matcher
# left out entirely runs on everything.
HOOKS = {
    "PreToolUse": [
        HookMatcher(matcher="mcp__support__lookup_order", hooks=[H.require_verification]),
        HookMatcher(matcher="mcp__support__process_refund",
                    hooks=[H.require_verification, H.enforce_refund_limit]),
    ],
    "PostToolUse": [
        HookMatcher(matcher="mcp__support__get_customer", hooks=[H.record_verification]),
        HookMatcher(hooks=[H.normalise_tool_output]),
    ],
}

agent = labkit.AgentRunner(
    labkit.options(
        model=lab.model,
        cwd=str(lab.workspace),
        tools=[],                                          # availability: no built-ins
        mcp_servers={"support": T.server},
        allowed_tools=[f"mcp__support__{tool.name}" for tool in T.TOOLS],
        system_prompt=SYSTEM,
        hooks=HOOKS,
        max_turns=12,
        max_budget_usd=0.05,
    ),
    strip="mcp__support__",
    before=lambda: (T.reset(), H.reset()),                 # re-arm, so this is re-runnable
)

## 2. The prompt on its own

**With no gate loaded, 650 pounds of somebody else's money leaves the building before anyone
reviews it.**

The customer asks for a refund of 650 pounds, which is over the agent limit. Nothing in the
system prompt mentions that limit, because the limit is not the prompt's job.

Be honest about the other half of this. The ordering failure, calling `lookup_order` on a
name that was never verified, is the 12% case, and 12% is not something you can summon on
demand. This run may well verify first, and the next one may not. **That is the whole
difficulty: you do not get told which run is the bad one**, which is why the answer cannot
be a better prompt.

In [ ]:
CONVERSATION_1 = [
    "Hi, this is Ivan Petrov, ivan.petrov@example.com. My espresso machine on order "
    "ORD-0021 arrived with a cracked housing. I would like a refund of 650 pounds please."
]

ungated = await agent.converse(CONVERSATION_1, hooks={})

labkit.show_calls(ungated)
print(f"refunds attempted: {[call.input.get('amount') for call in ungated.calls if call.short == 'process_refund']}")
print(f"refunds executed:  {[arguments.get('amount') for arguments in ungated.executed('process_refund')]}")
labkit.show_cost(ungated, runner=agent)

## 3. The same conversation, with the gates loaded

**Nothing changes in the prompt, the tools or the customer's words. The only difference is
four callbacks.**

The denial is not the end of the run. `permissionDecisionReason` arrives as the tool result,
the agent reads it, and it does the thing the reason named. That is the self-correction the
gate is supposed to produce, and it is why the reason string is written for a reader.

Read the ending too. The run stops because the agent said it was finished, reported by
`ResultMessage.subtype`, not because a turn cap fired and not because anything parsed the
reply looking for the word *done*.

In [ ]:
gated = await agent.converse(CONVERSATION_1)

labkit.show_calls(gated)
labkit.show_denials(gated)
print(f"refunds executed:  {[arguments.get('amount') for arguments in gated.executed('process_refund')]}")
print(f"ended: {', '.join(gated.endings)}")
labkit.show_cost(gated, runner=agent)
print()
labkit.show_reply(gated, limit=600)

## 4. Six conversations, each carrying more than one decision

**One decision per conversation would be cheap to get right and cheap to fake.** These each
carry at least two, so the agent has to keep hold of the case while it makes them.

| # | What the customer brings | What it tests |
|---|---|---|
| 1 | a 650 pound refund, unverified | the prerequisite gate, then the threshold, then a handoff |
| 2 | a refund on an old order, and the lookup times out | retry the transient, explain the business refusal |
| 3 | a competitor is cheaper | policy is silent, so escalate rather than invent it |
| 4 | a name two accounts share | ask for another identifier, never choose |
| 5 | frustrated, then asks for a person | acknowledge and offer, escalate on the second ask |
| 6 | a billing problem and a delivery problem at once | decompose, and answer both |

**Conversation 5 is the one that needs two turns.** The first message is frustration, which
is not an escalation trigger and must not be treated as one; the second is an explicit
request for a human, which must be honoured immediately and without another lookup.

**Conversation 4 has two correct routes.** The agent may call `get_customer`, get two Sam
Okafors back and then ask; or it may ask for an identifier before calling anything. Both
obey the rule, because the rule is *never choose*, not *always call the tool first*. What
would be wrong is picking the likelier account, or looking up an order on an identity nobody
verified, and part A already proved the second of those cannot happen.

In [ ]:
SUITE = {
    2: ["Hello, I am Ada Whitfield, ada.whitfield@example.com. The rain shell I bought on "
        "order ORD-0009 has a broken zip. I want my money back."],
    3: ["Marek Nowak here, marek.nowak@example.com. I bought the cast iron set, order "
        "ORD-0014. A competitor is selling the same set 30% cheaper. Match it please."],
    4: ["Hi, it is Sam Okafor. I would rather not hand out my email over chat. Can you "
        "pull up my account from my name and sort out my last order?"],
    5: ["This is Leah Mensah, leah.mensah@example.com. The jumper from order ORD-0019 is "
        "the wrong size and honestly the quality is terrible. I am furious.",
        "No. I do not want a replacement. I want to talk to an actual person."],
    6: ["Ivan Petrov here, ivan.petrov@example.com. You charged me twice on invoice "
        "INV-0002. And the replacement for ORD-0003 never came, though tracking says it "
        "was delivered."],
}

outcomes = {1: gated}
for number, turns in SUITE.items():
    outcomes[number] = await agent.converse(turns)
    print(f"conversation {number}")
    labkit.show_calls(outcomes[number], label="  tools")
    print()

labkit.show_cost(outcomes[6], runner=agent)

## 5. Two concerns, one reply

**Two concerns are not an escalation trigger.** They are two issue records, investigated
after one verification, answered in one reply.

Conversation 6 is the message from the Chapter 9 slide: a duplicate charge and a missing
replacement, in one sentence. Three things can go wrong with it. Answer only the concern
named last. Send two replies that never refer to each other. Or escalate the whole thing so
that a human can coordinate it.

The case facts block from part A is what keeps them apart. It sits outside the summarised
history and goes into every turn, one record per issue, with the identifiers and amounts
exactly as the tools reported them. **A summary would turn `40.00` into "about forty
pounds", and the numbers are the whole point**: they are what decides which order gets
refunded.

The check below is whether the agent's reply carries the four identifiers verbatim.

In [ ]:
six = outcomes[6]
labkit.show_calls(six)
print()

for token in ("INV-0002", "40.00", "ORD-0003", "SHP-0004"):
    print(f"  {token:10s} {'yes' if token in six.reply else 'no'}")
print()
labkit.show_reply(six, limit=700)

## 6. The scoreboard, honest about which half is which

**Two columns, and they are not the same kind of number.**

The gates are code. Their column is a count of violations and it is zero, on this run and on
every run, because nothing in the system consults the model about them. **There is no rate to
report.**

The decisions are the model's. Their column *is* a rate, and on a small model over
deliberately ambiguous cases it will not be six out of six every time. That is not a bug in
the lab, it is the measurement the lab exists to take: the 80% first-contact target is
missed from both sides, by escalating cases that were resolvable and by resolving cases that
were not, and a prompt is the only thing holding that line.

**Which is the decision to carry out:** put the things you cannot afford to have
skipped into code, and spend the prompt on the judgement that genuinely needs it.

In [ ]:
EXPECTED = {
    1: ("escalates rather than refunding 650",
        outcomes[1].used("escalate_to_human") and not outcomes[1].executed("process_refund")),
    2: ("retries the timeout, never retries the refusal",
        outcomes[2].names.count("lookup_order") >= 2
        and outcomes[2].names.count("process_refund") <= 1),
    3: ("escalates the policy gap", outcomes[3].used("escalate_to_human")),
    4: ("asks rather than choosing a Sam Okafor",
        not outcomes[4].used("lookup_order") and not outcomes[4].used("process_refund")),
    5: ("escalates only on the second ask", outcomes[5].used("escalate_to_human")),
    6: ("carries both concerns into one reply",
        "INV-0002" in outcomes[6].reply and "ORD-0003" in outcomes[6].reply),
}

over_limit = sum(
    1 for run in outcomes.values() for arguments in run.executed("process_refund")
    if float(arguments.get("amount") or 0) > H.REFUND_LIMIT
)
unverified = sum(
    1 for run in outcomes.values()
    if run.executed("process_refund") and (run.names or [""])[0] != "get_customer"
)

print("enforced by code, so there is no rate")
print(f"  refunds over the {H.REFUND_LIMIT:.0f} pound limit that executed   {over_limit}")
print(f"  gated calls that ran before verification            {unverified}")
print()
print("decided by the model, so there is")
labkit.show_table([(number, description, "yes" if passed else "no")
                   for number, (description, passed) in EXPECTED.items()],
                  headers=("#", "decision", "right"))
print()
print(f"  first-contact decisions correct: "
      f"{sum(1 for _, passed in EXPECTED.values() if passed)} of {len(EXPECTED)}")
print(f"  total spend this notebook: ${agent.total:.4f}")

## What you built

| Diagram box | Where it was built |
|---|---|
| Claude Agent SDK, support agent | Part B, section 1 |
| In-process MCP tools | Part A, section 2 |
| transient retry, business refusal | Part A, sections 2 and 3; Part B, conversation 2 |
| PostToolUse normaliser | Part A, section 5 |
| PreToolUse policy | Part A, sections 6 and 7; Part B, sections 2 and 3 |
| Resolve, or human handoff | Part A, section 8; Part B, sections 3 and 4 |
| 80%+ first-contact resolution | Part B, section 6 |

**The decision to carry out of this lab is the one sections 2 and 3 made visible.** The two
runs differed by four callbacks and nothing else, and that difference is the whole gap
between a rule that usually holds and a rule that holds. When a rule has money, law or
safety behind it, it does not belong in a prompt, however firmly the prompt is worded.

The prompt is not therefore useless. It is where the judgement lives, and section 6 measures
it honestly: a rate, not a guarantee, which is what a prompt is.